In [1]:
from pathlib import Path

import pandas as pd

# Carpeta con los CSV generados por `transform`
CSV_DIR = Path("../data/csv")

# Métricas a comparar entre el total del canal y la suma de vídeos
METRICS = ["views", "estimatedMinutesWatched"]

channel_daily = pd.read_csv(CSV_DIR / "analytics_channel_daily.csv")
video_totals = pd.read_csv(CSV_DIR / "analytics_video_totals.csv")
videos = pd.read_csv(CSV_DIR / "videos.csv", usecols=["video_id", "privacy_status"])

# Añade el estado de privacidad a los totales por vídeo.
# validate="one_to_one" falla si algún video_id estuviera repetido en alguna tabla.
video_totals = video_totals.merge(videos, on="video_id", how="left", validate="one_to_one")

# Comprobación previa: ningún vídeo de Analytics debe quedar sin estado de privacidad
sin_estado = int(video_totals["privacy_status"].isna().sum())
print(f"Vídeos de Analytics sin privacy_status: {sin_estado} (esperado: 0)\n")

# --- 3.1 Totales del canal (serie diaria) ----------------------------------
canal = channel_daily[METRICS].sum()

# --- 3.2 Suma de vídeos: todos, públicos y no públicos ---------------------
es_publico = video_totals["privacy_status"] == "public"
sumas = {
    "todos": video_totals[METRICS].sum(),
    "públicos": video_totals.loc[es_publico, METRICS].sum(),
    "no públicos": video_totals.loc[~es_publico, METRICS].sum(),
}

# --- 3.3 Tabla de comparación ----------------------------------------------
filas = []
for metrica in METRICS:
    total_canal = canal[metrica]
    fila = {"metrica": metrica, "canal": total_canal}
    for nombre, suma in sumas.items():
        fila[f"suma_{nombre}"] = suma[metrica]
    # Brecha = lo que el canal tiene y no se explica por los vídeos actuales
    fila["brecha_todos"] = total_canal - sumas["todos"][metrica]
    fila["brecha_todos_%"] = fila["brecha_todos"] / total_canal * 100
    fila["brecha_solo_publicos"] = total_canal - sumas["públicos"][metrica]
    fila["brecha_solo_publicos_%"] = fila["brecha_solo_publicos"] / total_canal * 100
    filas.append(fila)

comparacion = pd.DataFrame(filas).set_index("metrica")

print("== 3. Canal vs suma de vídeos ==")
print(comparacion.round(1).T.to_string())

# --- 3.4 Contexto: periodo cubierto por cada fuente ------------------------
print("\n== Periodo cubierto ==")
print(f"Serie diaria del canal: {channel_daily['day'].min()} -> {channel_daily['day'].max()}")
print(f"Vídeos con Analytics:   {len(video_totals)}")

Vídeos de Analytics sin privacy_status: 0 (esperado: 0)

== 3. Canal vs suma de vídeos ==
metrica                     views  estimatedMinutesWatched
canal                   4126269.0               11030983.0
suma_todos              3969064.0               10875512.0
suma_públicos           3894512.0               10798789.0
suma_no públicos          74552.0                  76723.0
brecha_todos             157205.0                 155471.0
brecha_todos_%                3.8                      1.4
brecha_solo_publicos     231757.0                 232194.0
brecha_solo_publicos_%        5.6                      2.1

== Periodo cubierto ==
Serie diaria del canal: 2015-04-11 -> 2026-10-20
Vídeos con Analytics:   371
